In [ ]:
from datasets import load_dataset

from tklearn.embeddings import AutoEmbedding

embedder = AutoEmbedding({
    "loader": "transformers",
    "name": "sentence-transformers/all-MiniLM-L6-v2",
})

In [ ]:
def get_embeddings(batch):
    return {
        "embeddings": embedder.encode(batch["sentence"]),
    }


dataset = load_dataset("glue", "sst2", split="train")
dataset = dataset.map(get_embeddings, batched=True, batch_size=10_000)

In [ ]:
dataset.set_format("numpy", columns=["embeddings"])

In [ ]:
dataset["embeddings"]

In [ ]:
from sentence_transformers import CrossEncoder

# 1. Load an STS (Semantic Similarity) Model
# This model is optimized to compare meaning, not search relevance.
model = CrossEncoder("cross-encoder/stsb-distilroberta-base")

target_word = "bank"
context_sentence = "He fished from the bank of the river."
definitions = [
    "A financial institution that accepts deposits.",
    "The land alongside or sloping down to a river or lake.",
    "To tilt an aircraft laterally in flight.",
]

# 2. Format the pairs
# Strategy: Compare the [Context] directly with the [Definition]
# We prefix the definition with the word to help the model link them.
pairs = []
for defn in definitions:
    # Pair: (Context Sentence,  Target Word + Definition)
    pairs.append([context_sentence, f"{target_word}: {defn}"])

# 3. Predict (Scores will be 0 to 1)
scores = model.predict(pairs)

# 4. Rank
results = list(zip(definitions, scores))
results.sort(key=lambda x: x[1], reverse=True)

print(f"Context: {context_sentence}\n")
for definition, score in results:
    print(f"Score: {score:.4f} | Def: {definition}")